# gVCF to VDS

In [1]:
# %%configure -f
# {
#   "driverMemory": "45G",
#   "conf": {
#     "spark.driver.memoryOverhead": "8G",
#     "spark.yarn.driver.memoryOverhead": "8G",
#     "spark.driver.maxResultSize": "4G",
#     "spark.speculation": "false"
#   }
# }

In [1]:
%%configure -f
{
    "driverMemory": "45G"
}

In [2]:
# Import and initiate HAIL
import hail as hl
hl.init(sc,log='/tmp/hail.log')

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1754891314340_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.5.2-amzn-1
SparkUI available at http://ip-192-168-125-185.ap-southeast-1.compute.internal:39305
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /tmp/hail.log

In [3]:
from pprint import pprint
pprint(dict(hl.spark_context().getConf().getAll()))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{'spark.app.attempt.id': '1',
 'spark.app.id': 'application_1754891314340_0001',
 'spark.app.name': 'livy-session-0',
 'spark.app.startTime': '1754892039706',
 'spark.app.submitTime': '1754892013913',
 'spark.blacklist.decommissioning.enabled': 'true',
 'spark.blacklist.decommissioning.timeout': '1h',
 'spark.decommissioning.timeout.threshold': '20',
 'spark.default.parallelism': '1008',
 'spark.driver.defaultJavaOptions': "-XX:OnOutOfMemoryError='kill -9 %p'",
 'spark.driver.extraClassPath': '/usr/local/lib/python3.9/site-packages/hail/backend/hail-all-spark.jar:/usr/lib/hadoop-lzo/lib/*:/usr/lib/hadoop/hadoop-aws.jar:/usr/share/aws/aws-java-sdk/*:/usr/share/aws/emr/emrfs/conf:/usr/share/aws/emr/emrfs/lib/*:/usr/share/aws/emr/emrfs/auxlib/*:/usr/share/aws/emr/goodies/lib/emr-spark-goodies.jar:/usr/share/aws/emr/security/conf:/usr/share/aws/emr/security/lib/*:/usr/share/aws/hmclient/lib/aws-glue-datacatalog-spark-client.jar:/usr/share/java/Hive-JSON-Serde/hive-openx-serde.jar:/usr/shar

## Load 1KG gVCF into VDS

- List the single sample hard-filtered.gvcf.gz generated for 1000 genomes dragen 3.7.6 analysis
- Upload the list (csv file) into S3
- Set gvcf_list_path

In [4]:
# gvcf_list_path='s3://npm-grids/hebrardms/batch/sg10k_reprocess_gvcf_manifest.csv'
# vds_prefix = 's3://precise-scratch/hebrardms/SG10K_Health/VDS'
gvcf_list_path='s3a://precise-scratch/goypav/1KG/VDS/1kg_gvcf_manifest.csv'
vds_prefix = 's3://precise-scratch/goypav/1KG/VDS'

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
# Import csv samples to ht
ht_gvcf = hl.import_table(gvcf_list_path, delimiter=',', quote = '"', no_header=True)
# Rename the columns
ht_gvcf = ht_gvcf.rename({'f0': 'bucket', 'f1': 'prefix'})
# Build S3 path
ht_gvcf = ht_gvcf.annotate(
    s3_path = hl.str('s3a://') + hl.str(ht_gvcf.bucket) + '/' + hl.str(ht_gvcf.prefix)
)

ht_gvcf.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

3205
2025-08-11 06:01:06.076 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)

In [6]:
# List of S3 path
ls_gvcf = ht_gvcf.s3_path.collect()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
## Test
###
gvcf_paths=ls_gvcf[0:10] # variant_data: 13905139 rows and 10 columns in 2586 partitions ~ 30Gb ~ 2h on 9 CPU onDemand
# gvcf_paths=ls_gvcf[0:100] # variant_data: 37755085 rows and 100 columns in 2586 partitions ~ 30Gb ~ 30min on 500 CPU onDemand
gvcf_paths

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['s3a://precise-ica-storage/byob/opendata-1000genomes/au-ilmn/ilmn-igg-1kgp/DRAGEN-3.7-gVCF/HG00096.hard-filtered.gvcf.gz', 's3a://precise-ica-storage/byob/opendata-1000genomes/au-ilmn/ilmn-igg-1kgp/DRAGEN-3.7-gVCF/HG00097.hard-filtered.gvcf.gz', 's3a://precise-ica-storage/byob/opendata-1000genomes/au-ilmn/ilmn-igg-1kgp/DRAGEN-3.7-gVCF/HG00099.hard-filtered.gvcf.gz', 's3a://precise-ica-storage/byob/opendata-1000genomes/au-ilmn/ilmn-igg-1kgp/DRAGEN-3.7-gVCF/HG00100.hard-filtered.gvcf.gz', 's3a://precise-ica-storage/byob/opendata-1000genomes/au-ilmn/ilmn-igg-1kgp/DRAGEN-3.7-gVCF/HG00101.hard-filtered.gvcf.gz', 's3a://precise-ica-storage/byob/opendata-1000genomes/au-ilmn/ilmn-igg-1kgp/DRAGEN-3.7-gVCF/HG00102.hard-filtered.gvcf.gz', 's3a://precise-ica-storage/byob/opendata-1000genomes/au-ilmn/ilmn-igg-1kgp/DRAGEN-3.7-gVCF/HG00103.hard-filtered.gvcf.gz', 's3a://precise-ica-storage/byob/opendata-1000genomes/au-ilmn/ilmn-igg-1kgp/DRAGEN-3.7-gVCF/HG00105.hard-filtered.gvcf.gz', 's3a://precise-

In [ ]:
# Step 1: gVCF combiner
###

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='20000')

# Combine gVCF
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/1000genomes_gvcf1000-bf50-tr100k-sp20k.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[0:1000],
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=50, # number of inputs combined in one VDS
    target_records=100000 # number of rows per partition
)

combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
# Step 2: gVCF combiner batch 2
###

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='20000')

# Combine next batch of gVCFs
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch2.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[1000:2000],
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=50,
    target_records=100000
)

combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-08-01 16:31:18.718 Hail: WARN: expected input file 's3a://precise-ica-storage/byob/opendata-1000genomes/au-ilmn/ilmn-igg-1kgp/DRAGEN-3.7-gVCF/HG02073.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-08-01 16:31:18.914 Hail: WARN: expected input file 's3a://precise-ica-storage/byob/opendata-1000genomes/au-ilmn/ilmn-igg-1kgp/DRAGEN-3.7-gVCF/HG02073.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-08-01 16:31:19.052 Hail: INFO: scanning VCF for sortedness...
2025-08-01 16:31:45.329 Hail: INFO: Coerced sorted VCF - no additional import work to do
2025-08-01 16:31:47.506 Hail: WARN: generated combiner save path of s3://precise-scratch/goypav/1KG/VDS/checkpoints/combiner-plans/vds-combiner-plan_b3bdb8efcd06e6f1796eb71c3f3a622be7407cdb2a7bf2624fffa93b100b5de6_0.2.134.json
2025-08-01 16:31:47.539 Hail: INFO: Running VDS combiner:
    VDS arguments: 0 datasets with 0 samples
    GVCF arguments: 1000 inputs/samples
    Branch factor: 50
    GVCF merge batch size: 50
2025-08-01 16:

In [ ]:
# Step 3: gVCF combiner batch 3
###

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='20000')

# Combine batch 3 gVCFs
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch3.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[2000:3000],
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=50,
    target_records=100000
)

combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Step 4: gVCF combiner batch 4 (final)
###

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='20000')

# Combine final batch of gVCFs
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch4.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[3000:3205],
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=50,
    target_records=100000
)

combiner.run()

In [ ]:
# # Step 5: VDS combiner
# ###

# # List of VDS
# # 2,000 samples -> 2 VDS of 1000 samples each
# ls_vds = [
#     "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k.vds",
#     "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch2.vds",
# ]

# # Change the number of tasks hail is launching in parallel
# hl._set_flags(spark_max_stage_parallelism='20000')

# # Combine next batch of gVCFs
# combiner = hl.vds.new_combiner(
#     output_path=f'{vds_prefix}/1000genomes_gvcf1000-bf50-tr100k-sp20k_combine_1_and_2.vds',
#     temp_path=f'{vds_prefix}/checkpoints/',
#     vds_paths=ls_vds,
#     use_genome_default_intervals=True,
#     reference_genome='GRCh38',
#     branch_factor=50,
#     target_records=100000
# )

# combiner.run()
# fail

In [8]:
# Use safer task parallelism (don't overload Spark)
hl._set_flags(spark_max_stage_parallelism='1000')  # reduce from 20000

# List of input VDS
ls_vds = [
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k.vds",
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch2.vds",
]

# Define output and temp paths (using s3:// to ensure Hail uses async boto I/O)
# vds_prefix = 's3://precise-scratch/goypav/1KG/VDS'

combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/1000genomes_combined_batch1_2.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    vds_paths=ls_vds,
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=20,           # reduce concurrency → deeper tree
    target_records=500_000      # reduce total number of partitions
)

# Run the combination job
combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-08-04 14:21:15.887 Hail: WARN: Mismatch between 'call_fields' and VDS call fields. Overwriting with call fields from supplied VDS.
    VDS call fields      : []
    requested call fields: ['PGT']

2025-08-04 14:21:15.951 Hail: WARN: generated combiner save path of s3://precise-scratch/goypav/1KG/VDS/checkpoints/combiner-plans/vds-combiner-plan_a3950e270f1e2bbdddb2de8d9c7de3832503d701097dcaece3d6b245140d9ae6_0.2.134.json
2025-08-04 14:21:23.284 Hail: INFO: Running VDS combiner:
    VDS arguments: 2 datasets with 2000 samples
    GVCF arguments: 0 inputs/samples
    Branch factor: 20
    GVCF merge batch size: 50
2025-08-04 14:21:23.447 Hail: INFO: VDS Combine (job 1): merging 2 datasets with 2000 samples
2025-08-04 14:50:36.982 Hail: INFO: wrote table with 2876510599 rows in 24273 partitions to s3://precise-scratch/goypav/1KG/VDS/checkpoints/combiner-intermediates/ddd23ed2-0b47-49da-9500-118e27e42819_vds-combine_job1/interval_checkpoint.ht
2025-08-04 16:39:02.032 Hail: INFO: wrote 

In [ ]:
# Use safer task parallelism (don't overload Spark)
hl._set_flags(spark_max_stage_parallelism='1000')  # reduce from 20000

# List of input VDS
ls_vds = [
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k.vds",
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch2.vds",
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch3.vds",
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch4.vds",
    
]

# Define output and temp paths (using s3:// to ensure Hail uses async boto I/O)
# vds_prefix = 's3://precise-scratch/goypav/1KG/VDS'

combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/1000genomes_gvcf1000-bf50-tr100k-sp20k_vds4-bf2-tr50k-sp1k.n3205.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    vds_paths=ls_vds,
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=2,           # reduce concurrency → deeper tree
    target_records=50_000      # reduce total number of partitions # with 500k 5x1000 ok but 2 out of 755 task lasts forever 
)

# Run the combination job
combiner.run()

In [ ]:
# Safe parallelism setting to prevent scheduler overload
hl._set_flags(spark_max_stage_parallelism='1000')

# Input VDS paths
ls_vds = [
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k.vds",
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch2.vds",
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch3.vds",
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch4.vds",
]

# VDS combiner config with reduced branch factor
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/1000genomes_combined_vds4-bf10-tr1m-sp1k.n3205.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    vds_paths=ls_vds,
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=10,              # less concurrency, deeper tree
    target_records=1_000_000         # larger partitions = more efficient
)

# Run
combiner.run()


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
# Use safer task parallelism (don't overload Spark)
hl._set_flags(spark_max_stage_parallelism='1000')  # reduce from 20000

# List of input VDS
ls_vds = [
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch3.vds",
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch4.vds",
]

# Define output and temp paths (using s3:// to ensure Hail uses async boto I/O)
# vds_prefix = 's3://precise-scratch/goypav/1KG/VDS'

combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/1000genomes_combined_batch3_4.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    vds_paths=ls_vds,
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=20,           # reduce concurrency → deeper tree
    target_records=500_000      # reduce total number of partitions
)

# Run the combination job
combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-08-07 13:00:26.069 Hail: WARN: Mismatch between 'call_fields' and VDS call fields. Overwriting with call fields from supplied VDS.
    VDS call fields      : []
    requested call fields: ['PGT']

2025-08-07 13:00:27.087 Hail: WARN: generated combiner save path of s3://precise-scratch/goypav/1KG/VDS/checkpoints/combiner-plans/vds-combiner-plan_ac1063194e758ad41e8f64a1fe0df2826f8ad5888f596f4040fb7a920b7545c2_0.2.134.json
2025-08-07 13:00:36.611 Hail: INFO: Running VDS combiner:
    VDS arguments: 2 datasets with 1205 samples
    GVCF arguments: 0 inputs/samples
    Branch factor: 20
    GVCF merge batch size: 50
2025-08-07 13:00:36.820 Hail: INFO: VDS Combine (job 1): merging 2 datasets with 1205 samples
2025-08-07 13:32:31.950 Hail: INFO: wrote table with 2877467756 rows in 25310 partitions to s3://precise-scratch/goypav/1KG/VDS/checkpoints/combiner-intermediates/48d11e81-e703-4318-a175-e9ad500f9517_vds-combine_job1/interval_checkpoint.ht
2025-08-07 15:07:52.460 Hail: INFO: wrote 

In [8]:
# Use safer task parallelism (don't overload Spark)
hl._set_flags(spark_max_stage_parallelism='1000')  # reduce from 20000

# List of input VDS
ls_vds = [
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_combined_batch1_2.vds",
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_combined_batch3_4.vds",
]

# Define output and temp paths (using s3:// to ensure Hail uses async boto I/O)
# vds_prefix = 's3://precise-scratch/goypav/1KG/VDS'

combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/1000genomes_combined_batch1_2_batch3_4.n3205.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    vds_paths=ls_vds,
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=20,           # reduce concurrency → deeper tree
    target_records=500_000      # reduce total number of partitions
)

# Run the combination job
combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-08-08 01:51:55.282 Hail: WARN: Mismatch between 'call_fields' and VDS call fields. Overwriting with call fields from supplied VDS.
    VDS call fields      : []
    requested call fields: ['PGT']

2025-08-08 01:51:56.246 Hail: WARN: generated combiner save path of s3://precise-scratch/goypav/1KG/VDS/checkpoints/combiner-plans/vds-combiner-plan_650b4d3b6713d9c2b27416b1ed3e22dbdb1e0c2d6161b9e9d11c1feece882e2b_0.2.134.json
2025-08-08 01:52:01.506 Hail: INFO: Running VDS combiner:
    VDS arguments: 2 datasets with 3205 samples
    GVCF arguments: 0 inputs/samples
    Branch factor: 20
    GVCF merge batch size: 50
2025-08-08 01:52:01.671 Hail: INFO: VDS Combine (job 1): merging 2 datasets with 3205 samples
2025-08-08 02:38:13.385 Hail: INFO: wrote table with 2882690172 rows in 5755 partitions to s3://precise-scratch/goypav/1KG/VDS/checkpoints/combiner-intermediates/b583028b-c63b-4f63-9348-6ec72f676c6b_vds-combine_job1/interval_checkpoint.ht
2025-08-08 04:13:57.574 Hail: INFO: wrote m

In [8]:
# Use safer task parallelism (don't overload Spark)
hl._set_flags(spark_max_stage_parallelism='1000')  # reduce from 20000

# List of input VDS
ls_vds = [
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k.vds",
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch2.vds",
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch3.vds",
    "s3a://precise-scratch/goypav/1KG/VDS/1000genomes_gvcf1000-bf50-tr100k-sp20k_batch4.vds",
]

# Define output and temp paths (using s3:// to ensure Hail uses async boto I/O)
# vds_prefix = 's3://precise-scratch/goypav/1KG/VDS'

combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/1000genomes_combined_batch1_2_3_4.bf2-tr500k-sp1k.n3205.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    vds_paths=ls_vds,
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=2,           # reduce concurrency → deeper tree
    target_records=500_000      # reduce total number of partitions
)

# Run the combination job
combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-08-11 06:04:49.805 Hail: WARN: Mismatch between 'call_fields' and VDS call fields. Overwriting with call fields from supplied VDS.
    VDS call fields      : []
    requested call fields: ['PGT']

2025-08-11 06:04:50.730 Hail: WARN: generated combiner save path of s3://precise-scratch/goypav/1KG/VDS/checkpoints/combiner-plans/vds-combiner-plan_1fe73ff525fe61d8315000fab5bfc49a7d3548b52a550bc11e1ffa416bd46ac4_0.2.134.json
2025-08-11 06:05:09.861 Hail: INFO: Running VDS combiner:
    VDS arguments: 4 datasets with 3205 samples
    GVCF arguments: 0 inputs/samples
    Branch factor: 2
    GVCF merge batch size: 50
2025-08-11 06:05:10.060 Hail: INFO: VDS Combine (job 1): merging 2 datasets with 1205 samples
2025-08-11 06:37:18.399 Hail: INFO: wrote table with 2877467756 rows in 25310 partitions to s3://precise-scratch/goypav/1KG/VDS/checkpoints/combiner-intermediates/f4d60fd7-76e9-412b-bfde-036c6648df2f_vds-combine_job1/interval_checkpoint.ht
2025-08-11 08:09:43.365 Hail: INFO: wrote m

In [ ]:
# Step 2: VDS combiner
###

# List of VDS
# 1,000 samples -> 20 VDS of 50 samples each
ls_vds = [
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_00.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_01.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_02.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_03.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_04.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_05.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_06.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_07.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_08.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_09.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_10.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_11.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_12.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_13.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_14.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_15.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_16.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_17.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_18.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_19.vds",
]

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='30000')

# Combine VDS
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_1k-sp30k-bf5-tr30k.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    vds_paths=ls_vds,
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=5, # number of inputs combined in one VDS
    target_records=30000 # number of rows per partition
)
combiner.run()

In [ ]:
# Step 3: VDS combiner
###

# List of VDS
# 1,000 samples -> 4 VDS of 250 samples each
ls_vds = [
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/cc8c8f29-3267-499d-a143-88253ffb203b_vds-combine_job1/dataset.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/cc8c8f29-3267-499d-a143-88253ffb203b_vds-combine_job2/dataset.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/cc8c8f29-3267-499d-a143-88253ffb203b_vds-combine_job3/dataset.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/cc8c8f29-3267-499d-a143-88253ffb203b_vds-combine_job4/dataset.vds",
]

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='20000')

# Combine VDS
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_1k-sp20k-bf2-tr30k.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    vds_paths=ls_vds,
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=2, # number of inputs combined in one VDS
    target_records=30000 # number of rows per partition
)
combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Check existing VDS

In [ ]:
base_uri = 's3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/2f09aa4a-5bd6-4c00-bfad-9dbac316451b_vds-combine_job1/'
interval = 'interval_checkpoint.ht'
ref = 'dataset.vds/reference_data'
var = 'dataset.vds/variant_data'

ht_interval = hl.read_table(f'{base_uri}{interval}')
print(f'interval_checkpoint: {ht_interval.n_partitions():,}')
print(ht_interval.count())

mt_ref = hl.read_matrix_table(f'{base_uri}{ref}')
print(f'reference_data: {mt_ref.n_partitions():,}')
print(mt_ref.count())

mt_var = hl.read_matrix_table(f'{base_uri}{var}')
print(f'variant_data: {mt_var.n_partitions():,}')
print(mt_var.count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

reference_data: 2,586
(2837116404, 50)
variant_data: 2,586
(26852099, 50)

In [ ]:
vds_uri = 's3://precise-scratch/hebrardms/SG10K_Health/VDS/SG10K_Health_1k-sp20k-bf2-tr30k.vds/'
vds_ref = 'reference_data'
vds_var = 'variant_data'

mt_ref = hl.read_matrix_table(f'{vds_uri}{vds_ref}')
print(f'reference_data: {mt_ref.n_partitions():,}')
print(mt_ref.count())

mt_var = hl.read_matrix_table(f'{vds_uri}{vds_var}')
print(f'variant_data: {mt_var.n_partitions():,}')
print(mt_var.count())